In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
def split_et_scale_proprement_sans_stratify(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test


def charger_immobilier():
    data = fetch_california_housing()
    X = data.data
    y = data.target

    print(
        f"California Housing : {X.shape} variables, "
        "cible = prix médian en centaines de milliers de $"
    )
    print("\n Features :")
    print(data.feature_names)

    return X, y

def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)

    y_pred = modele.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)

    return {
        "r2": r2,
        "mae": mae,
        "rmse": rmse
    }

def checkpoint_qualite(X, y, case = "normal"):
    if case == "limite":
        X_small = X[:100]
        y_small = y[:100]
        return X_small, y_small
    
    return X, y

def predict_prix_immobilier(X, y, case = "normal"):
    X_checked, y_checked = checkpoint_qualite(X, y, case)

    X_train_scaled, X_test_scaled, y_train, y_test = split_et_scale_proprement_sans_stratify(X_checked, y_checked)

    lr = LinearRegression()

    resultats_lr = evaluer_regression(
        lr,
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test
    )

    print(
        f"\n LinearRegression : "
        f"R2={resultats_lr['r2']:.2f} "
        f"MAE={resultats_lr['mae']:.2f} "
        f"RMSE={resultats_lr['rmse']:.2f}"
    )

    rf = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    resultats_rf = evaluer_regression(
        rf,
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test
    )

    print(
        f"\n RandomForest : "
        f"R2={resultats_rf['r2']:.2f} "
        f"MAE={resultats_rf['mae']:.2f} "
        f"RMSE={resultats_rf['rmse']:.2f}"
    )

In [ ]:
print("======Phase A : Prédire les prix immobiliers (régression)=========")

X, y = charger_immobilier()
print("======CAS NORMAL=========")
predict_prix_immobilier(X, y)

print("======CAS LIMITE=========")
predict_prix_immobilier(X, y, "limite")

print("======CAS ADVERSARIAL=========")
X_fictif = np.array([[
    0.0,
    20.0,
    5.0,
    1.0,
    9000.0,
    4.0,
    34.0,
    -118.0
]])

y_fictif = np.array([0])